# 13-7. 프로젝트 B — Windows 아티팩트 통합 분석 예제

## Goal

- 정규화 레코드에 검토 규칙을 적용합니다.
- 검토 항목을 악성 확정과 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

13-6과 같은 합성 사건을 임시 디렉터리에서 다시 생성합니다.


## Steps

### 통합 검토 항목 생성

제공 파이프라인의 `review_findings()`로 근거 UID가 연결된 검토 항목을 만듭니다.


In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
from tempfile import TemporaryDirectory


def project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples/13-kape-triage/pipeline.py").is_file():
            return candidate
    raise RuntimeError("저장소 루트에서 Notebook을 실행해야 합니다")


def load_module(name: str, path: Path):
    spec = spec_from_file_location(name, path)
    module = module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


MODULE_ROOT = project_root() / "examples/13-kape-triage"
sample_module = load_module("chapter13b_make_sample", MODULE_ROOT / "make_sample.py")
pipeline_module = load_module("chapter13b_pipeline", MODULE_ROOT / "pipeline.py")

with TemporaryDirectory() as directory:
    manifest_path = sample_module.make_sample(Path(directory) / "input")
    _, _, events, issues, coverage = pipeline_module.load_case(manifest_path)
    findings, related_paths = pipeline_module.review_findings(events)
    known_uids = {event["uid"] for event in events}
    project_b_summary = {
        "findings": len(findings),
        "related_paths": len(related_paths),
        "priorities": sorted({finding["priority"] for finding in findings}),
        "all_evidence_linked": all(set(finding["event_uids"]) <= known_uids for finding in findings),
    }
print(project_b_summary)
for finding in findings[:3]:
    print(finding["rule_id"], finding["title"], finding["status"])


{'findings': 8, 'related_paths': 2, 'priorities': [20, 30, 40], 'all_evidence_linked': True}
LAB-AUTH-01 5분 내 반복 로그온 실패 검토 review
LAB-AUTORUN-01 Run/RunOnce 값 검토 review
LAB-PROCESS-01 사용자 쓰기 가능 경로의 프로세스 생성 검토 review


## Checks

검토 항목 수와 근거 UID 연결을 확인합니다.


In [2]:
assert project_b_summary["findings"] == 8
assert project_b_summary["related_paths"] == 2
assert project_b_summary["all_evidence_linked"] is True
assert all(finding["status"] == "review" for finding in findings)
assert all(finding["event_uids"] for finding in findings)
print("프로젝트 B 기준 검사 통과")


프로젝트 B 기준 검사 통과


## Next Steps

규칙 일치는 조사 우선순위를 좁히는 단서이며 원본 사건·승인 이력·추가 아티팩트와 대조해야 합니다.
